In [17]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import (
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)
from sklearn.model_selection import StratifiedKFold

In [18]:
# Load Welsh CEFR dataset from HuggingFace
ds = load_dataset("UniversalCEFR/learn_welsh_cy")["train"]
df_main = ds.to_pandas()  

# Load your B2 JSON data
df_b2 = pd.read_json("b2_welsh.json")
df_b2["cefr_level"] = "B2"
df_b2 = df_b2.drop_duplicates(subset="text", keep="first")

# Combine both DataFrames
df_combined = pd.concat([df_main, df_b2], ignore_index=True)
df_combined = df_combined.drop_duplicates(subset="text", keep="first")

# Convert back to HuggingFace Dataset
ds_merged = Dataset.from_pandas(df_combined)

In [19]:
ds_merged

Dataset({
    features: ['title', 'lang', 'source_name', 'format', 'category', 'cefr_level', 'license', 'text', '__index_level_0__'],
    num_rows: 2020
})

In [20]:
CEFR_LEVELS = ["A1", "A2", "B1", "B2", "C1", "C2"]
label2id = {lvl: i for i,lvl in enumerate(CEFR_LEVELS)}
labels = np.array([label2id[l] for l in ds_merged["cefr_level"]])

In [21]:
model_name = "./eurobert_cefr_english_only/final_model"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, trust_remote_code=True)
data_collator = DataCollatorWithPadding(tokenizer)

In [22]:
def preprocess(batch):
    toks = tokenizer(batch["text"], truncation=True, max_length=256)
    toks["labels"] = [label2id[l] for l in batch["cefr_level"]]
    return toks

In [25]:
# Metrics
def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, labels=list(range(len(CEFR_LEVELS))), zero_division=0
    )

    metrics = {}
    for i, label in enumerate(CEFR_LEVELS):
        metrics[f"{label}_precision"] = precision[i]
        metrics[f"{label}_recall"] = recall[i]
        metrics[f"{label}_f1"] = f1[i]

    metrics["eval_accuracy"] = accuracy_score(labels, preds)
    metrics["eval_weighted_f1"] = f1_score(labels, preds, average="weighted")
    metrics["eval_weighted_precision"] = precision_score(labels, preds, average="weighted")
    metrics["eval_weighted_recall"] = recall_score(labels, preds, average="weighted")
    return metrics

In [26]:
# Cross-validation setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
all_results = []

In [27]:
best_f1 = 0.0
best_trainer = None
best_tokenizer = None

for fold, (train_idx, val_idx) in enumerate(skf.split(ds_merged, labels), start=1):
    print(f"\n Running Fold {fold}...")

    ds_train = ds_merged.select(train_idx)
    ds_val = ds_merged.select(val_idx)

    tok_train = ds_train.map(preprocess, batched=True, remove_columns=ds_train.column_names)
    tok_val = ds_val.map(preprocess, batched=True, remove_columns=ds_val.column_names)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, trust_remote_code=True)

    args = TrainingArguments(
        output_dir=f"./eurobert_cefr_english_welsh_b2/fold_{fold}",  
        num_train_epochs=3, 
        per_device_train_batch_size=2,              
        per_device_eval_batch_size=3,                
        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_weighted_f1",
        greater_is_better=True,
        seed=42,
        learning_rate=3.6e-5,
        warmup_ratio=0.1,
        gradient_accumulation_steps=16,      
        optim="adamw_torch_fused",                   
        lr_scheduler_type="linear",                  
        adam_beta1=0.9,
        adam_beta2=0.999,
        adam_epsilon=1e-8,
        save_total_limit=1,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tok_train,
        eval_dataset=tok_val,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    metrics = trainer.evaluate()

    # Track best trainer
    if metrics["eval_weighted_f1"] > best_f1:
        best_f1 = metrics["eval_weighted_f1"]
        best_trainer = trainer
        best_tokenizer = tokenizer

    # Store fold metrics
    row = {
        "Fold": fold,
        "All CEFR Levels Precision": metrics.get("eval_weighted_precision", 0.0),
        "All CEFR Levels Recall": metrics.get("eval_weighted_recall", 0.0),
        "All CEFR Levels F1": metrics.get("eval_weighted_f1", 0.0),
    }
    for level in ["A1", "A2", "B1", "B2", "C1", "C2"]:
        row[f"{level} Precision"] = metrics.get(f"eval_{level}_precision", 0.0)
        row[f"{level} Recall"] = metrics.get(f"eval_{level}_recall", 0.0)
        row[f"{level} F1"] = metrics.get(f"eval_{level}_f1", 0.0)

    all_results.append(row)


 Running Fold 1...


Map: 100%|██████████| 404/404 [00:00<00:00, 10349.10 examples/s]
C:\Users\c24082331\AppData\Local\Temp\ipykernel_43580\2116516503.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B1 Precision,B1 Recall,B1 F1,B2 Precision,B2 Recall,B2 F1,C1 Precision,C1 Recall,C1 F1,C2 Precision,C2 Recall,C2 F1
1,1.181100,0.955314,0.547030,0.537654,0.630107,0.547030,0.615385,0.784314,0.689655,0.368098,0.495868,0.422535,0.000000,0.000000,0.000000,0.891304,0.315385,0.465909,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.702600,0.671083,0.707921,0.706386,0.741546,0.707921,0.755952,0.830065,0.791277,0.569620,0.743802,0.645161,0.000000,0.000000,0.000000,0.884615,0.530769,0.663462,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.385000,0.576317,0.792079,0.790908,0.790985,0.792079,0.817073,0.875817,0.845426,0.741379,0.710744,0.725738,0.000000,0.000000,0.000000,0.806452,0.769231,0.787402,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000



 Running Fold 2...


Map: 100%|██████████| 404/404 [00:00<00:00, 14533.20 examples/s]
C:\Users\c24082331\AppData\Local\Temp\ipykernel_43580\2116516503.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


...

Map: 100%|██████████| 404/404 [00:00<00:00, 10797.66 examples/s]
C:\Users\c24082331\AppData\Local\Temp\ipykernel_43580\2116516503.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B1 Precision,B1 Recall,B1 F1,B2 Precision,B2 Recall,B2 F1,C1 Precision,C1 Recall,C1 F1,C2 Precision,C2 Recall,C2 F1
1,1.132600,0.844928,0.559406,0.564258,0.637088,0.559406,0.797297,0.388158,0.522124,0.364583,0.578512,0.447284,0.000000,0.000000,0.000000,0.702899,0.740458,0.721190,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.657200,0.585371,0.745050,0.743528,0.768812,0.745050,0.788571,0.907895,0.844037,0.594406,0.702479,0.643939,0.000000,0.000000,0.000000,0.906977,0.595420,0.718894,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.299800,0.481726,0.839109,0.840251,0.843944,0.839109,0.882353,0.888158,0.885246,0.736842,0.809917,0.771654,0.000000,0.000000,0.000000,0.898305,0.809160,0.851406,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000



 Running Fold 5...


Map: 100%|██████████| 404/404 [00:00<00:00, 9968.81 examples/s]
C:\Users\c24082331\AppData\Local\Temp\ipykernel_43580\2116516503.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B1 Precision,B1 Recall,B1 F1,B2 Precision,B2 Recall,B2 F1,C1 Precision,C1 Recall,C1 F1,C2 Precision,C2 Recall,C2 F1
1,1.204800,0.942104,0.547030,0.538628,0.533975,0.547030,0.601156,0.684211,0.640000,0.350000,0.289256,0.316742,0.000000,0.000000,0.000000,0.625954,0.625954,0.625954,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.834500,0.872326,0.601485,0.572986,0.576135,0.601485,0.632558,0.894737,0.741144,0.422535,0.247934,0.312500,0.000000,0.000000,0.000000,0.652542,0.587786,0.618474,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.690000,0.765311,0.663366,0.658666,0.656129,0.663366,0.761290,0.776316,0.768730,0.500000,0.438017,0.466960,0.000000,0.000000,0.000000,0.678322,0.740458,0.708029,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [28]:
# Save best-performing model from all folds
final_path = "./eurobert_cefr_english_welsh_b2/best_model"
best_trainer.save_model(final_path)
best_tokenizer.save_pretrained(final_path)
best_trainer.state.save_to_json(os.path.join(final_path, "trainer_state.json"))

In [29]:
# Convert to DataFrame
df = pd.DataFrame(all_results)

# Compute average row
average_row = df.drop(columns=["Fold"]).mean(numeric_only=True)
average_row["Fold"] = "Average"
df = pd.concat([df, pd.DataFrame([average_row])], ignore_index=True)

# Restructure columns
columns = [("Fold", "")] + [
    ("All CEFR Levels", "Precision"), ("All CEFR Levels", "Recall"), ("All CEFR Levels", "F1"),
    ("A1", "Precision"), ("A1", "Recall"), ("A1", "F1"),
    ("A2", "Precision"), ("A2", "Recall"), ("A2", "F1"),
    ("B1", "Precision"), ("B1", "Recall"), ("B1", "F1"),
    ("B2", "Precision"), ("B2", "Recall"), ("B2", "F1"),
    ("C1", "Precision"), ("C1", "Recall"), ("C1", "F1"),
    ("C2", "Precision"), ("C2", "Recall"), ("C2", "F1"),
]


df = df[[col[0] if col[1] == "" else f"{col[0]} {col[1]}" for col in columns]]
df.columns = pd.MultiIndex.from_tuples(columns)

In [30]:
df

Fold All CEFR Levels                            A1                      \
                 Precision    Recall        F1 Precision    Recall        F1   
0        1        0.790985  0.792079  0.790908  0.817073  0.875817  0.845426   
1        2        0.794742  0.794554  0.793941  0.850000  0.888889  0.869010   
2        3        0.788457  0.789604  0.787008  0.802326  0.907895  0.851852   
3        4        0.843944  0.839109  0.840251  0.882353  0.888158  0.885246   
4        5        0.656129  0.663366  0.658666  0.761290  0.776316  0.768730   
5  Average        0.774852  0.775743  0.774155  0.822608  0.867415  0.844053   

         A2                      ...   B1        B2                      \
  Precision    Recall        F1  ...   F1 Precision    Recall        F1   
0  0.741379  0.710744  0.725738  ...  0.0  0.806452  0.769231  0.787402   
1  0.712000  0.735537  0.723577  ...  0.0  0.806723  0.738462  0.771084   
2  0.750000  0.688525  0.717949  ...  0.0  0.808333  0.746154  0.776000   
3  0.736842  0.809917  0.771654  ...  0.0  0.898305  0.809160  0.851406   
4  0.500000  0.438017  0.466960  ...  0.0  0.678322  0.740458  0.708029   
5  0.688044  0.676548  0.681176  ...  0.0  0.799627  0.760693  0.778784   

         C1                    C2              
  Precision Recall   F1 Precision Recall   F1  
0       0.0    0.0  0.0       0.0    0.0  0.0  
1       0.0    0.0  0.0       0.0    0.0  0.0  
2       0.0    0.0  0.0       0.0    0.0  0.0  
3       0.0    0.0  0.0       0.0    0.0  0.0  
4       0.0    0.0  0.0       0.0    0.0  0.0  
5       0.0    0.0  0.0       0.0    0.0  0.0  

[6 rows x 22 columns]